# Graph Matching


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
CODE_DIR = PROJECT_DIR / "code"
sys.path.insert(0, str(CODE_DIR))

PROJECT_DIR, CODE_DIR

In [ ]:
import pandas as pd

from graph_matching import load_edges, run_approach

DATA_PATH = PROJECT_DIR / "dataset" / "email-Eu-core-temporal.txt"
OUTPUT_DIR = CODE_DIR / "outputs" / "graph_matching"

edges = load_edges(DATA_PATH, cutoff_days=500)
edges.head(), len(edges), edges["ts"].max() / (24 * 60 * 60)

In [ ]:
for approach in ["cumulative", "interval", "overlap"]:
    run_approach(
        edges=edges,
        approach=approach,
        output_dir=OUTPUT_DIR,
        cutoff_days=500,
        snapshot_days=50,
        num_snapshots=10,
        overlap_fraction=0.5,
        seed=42,
        resolution=1.0,
        min_community_size=3,
        match_threshold=0.3,
    )

OUTPUT_DIR

In [ ]:
summary = []
for approach in ["cumulative", "interval", "overlap"]:
    base = OUTPUT_DIR / approach
    stats = pd.read_csv(base / "snapshot_stats.csv")
    communities = pd.read_csv(base / "communities.csv")
    matches = pd.read_csv(base / "matches.csv")
    events = pd.read_csv(base / "events.csv")
    summary.append({
        "approach": approach,
        "snapshots": len(stats),
        "communities": len(communities),
        "matches": len(matches),
        "births": int((events["event_type"] == "birth").sum()),
        "deaths": int((events["event_type"] == "death").sum()),
        "splits": int((events["event_type"] == "split").sum()),
        "merges": int((events["event_type"] == "merge").sum()),
    })

pd.DataFrame(summary)